# Loading in our files:

In [ ]:
import os
import json
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm.notebook import tqdm

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [ ]:
files = sorted(Path("processed_chunks").glob("chunk_*.pkl"))

for file in files[:3]:
    df = pd.read_pickle(file)
    
    print(file.name)
    print(df.head())

# Loading and splitting data:

In [ ]:
fake_categories = {"fake", "hate", "unreliable", "clickbait", "conspiracy", "junksci", "rumor"}
valid_types = fake_categories | {"reliable", "political", "bias"}

def load_chunks_logistic(files):
    dfs = []
    for file in tqdm(files):
        df = pd.read_pickle(file)
        df = df[df["type"].isin(valid_types)][["type", "tokens_stemmed"]]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    del dfs
    data["text"] = data["tokens_stemmed"].apply(lambda tokens: " ".join(tokens))
    data["label"] = data["type"].apply(lambda x: 1 if x in fake_categories else 0)
    return data[["text", "label"]]

train_data = load_chunks_logistic(files[:80])
val_data   = load_chunks_logistic(files[80:90])
test_data  = load_chunks_logistic(files[90:100])

X_train, y_train = train_data["text"], train_data["label"]
X_val,   y_val   = val_data["text"],   val_data["label"]
X_test,  y_test  = test_data["text"],  test_data["label"]


# Simple logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

# Vectorize
vectorizer = TfidfVectorizer(max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec   = vectorizer.transform(X_val)
X_test_vec  = vectorizer.transform(X_test)

# Train
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate on validation set
y_val_pred = model.predict(X_val_vec)
print("Validation:")
print(classification_report(y_val, y_val_pred, target_names=["reliable", "fake"]))